## 📑 Core Legal Process Assignment
 
#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/AQCkHg)**  
  Comprehensive documentation outlining the business requirements for this report.

##### Authentication  

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col,regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()

import pandas as pd
from pyspark.sql import DataFrame
from pyspark.sql.functions import trim
import boto3
import os

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d') 

In [0]:
table_name = 'Core_LegalProcessAssignment'

schema = (
    "BusinessDate date,"
    "CIF string,"
    "AFM string,"
    "CUST_LAST_NAME string,"
    "CUST_FIRST_NAME string,"
    "CUST_FATH_NAME string,"
    "ACCOUNT string,"
    "PRIMARY_OWNER_ACCOUNT_NUMBER string,"
    "QC_RELATION_TYPE string,"
    "ACCOUNT_STATUS string,"
    "CURRENT_BUCKET string,"
    "CASE_OUTST_BALANCE string,"
    "CASE_DEBT_AMOUNT string,"
    "DEBT_LPS string,"
    "DEBT_LPS_DATE string,"
    "CUS_LPS string,"
    "CurrentAgencyPaymentOrder string,"
    "CurrentAgencyAssignmentDate string,"
    "CurrentStagePaymentOrder string,"
    "CurrentStageDatePaymentOrder string,"
    "CurrentAgencyForcedPrenotation string,"
    "CurrentAgencyAssignmentDateForcedPrenotation string,"
    "CurrentStageForcedPrenotation string,"
    "CurrentStageDateForcedPrenotation string,"
    "CurrentAgencyConfiscation string,"
    "CurrentAgencyAssignmentDateconfiscation string,"
    "CurrentStageconfiscation string,"
    "CurrentStageDateconfiscation string,"
    "ON_BALANCE string,"
    "OFF_BALANCE string,"
    "TOTAL_BALANCE string,"
    "TOTAL_BALANCE_EMLT string,"
    "CurrentAgencyRegularSuit string,"
    "CurrentAgencyAssignmentDateRegularSuit string,"
    "CurrentStageRegularSuit string,"
    "CurrentStageDateRegularSuit string,"
    "InsertTimeStamp timestamp"	
    )

In [0]:
partition_columns = ["BusinessDate"]
delivery_table_path = ""
 
# ================== Create Delta Table ==================
# env_prefix ='qa' #IL: To be Deleted!

delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"
delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_date,  # type: ignore
)


##### Create Delta Tables with selected fields & Temporary Views for each delta table

##### NonCorePerimeter

In [0]:
NonCorePerimeter = spark.sql(
f'''
SELECT ExposureKeyQCAlpha
     , ExposureKey
     , BusinessDate
FROM `{env_prefix}-cdh`.derivative.dexposure
WHERE 1=1
AND PortfolioName <> 'Core Retail'
'''
)
NonCorePerimeter.createOrReplaceTempView("NonCorePerimeter")

##### LegalProcessAssignment

In [0]:
LegalProcessAssignment = spark.sql(
f'''
    SELECT distinct

          '{run_date}' AS BusinessDate
          ,a.CIF
          ,a.AFM
          ,a.CUST_LAST_NAME
          ,a.CUST_FIRST_NAME
          ,a.CUST_FATH_NAME
          ,a.ACCOUNT
          ,a.PRIMARY_OWNER_ACCOUNT_NUMBER
          ,a.`QC RELATION TYPE` AS QC_RELATION_TYPE --change IL
          ,a.ACCOUNT_STATUS 
          ,a.CURRENT_BUCKET
          ,a.CASE_OUTST_BALANCE
          ,a.CASE_DEBT_AMOUNT
          ,a.DEBT_LPS
          ,a.DEBT_LPS_DATE
          ,NULL AS CUS_LPS
          ,a.`Current Agency Διαταγής Πληρωμής`                           AS CurrentAgencyPaymentOrder
          ,a.`Current Agency Assignment Date`                             AS CurrentAgencyAssignmentDate
          ,a.`Current Stage Διαταγής Πληρωμής`                            AS CurrentStagePaymentOrder
          ,a.`Current Stage Date Διαταγής Πληρωμής`                       AS CurrentStageDatePaymentOrder
          ,a.`Current Agency Αναγκαστικής Προσημείωσης`                  AS CurrentAgencyForcedPrenotation
          ,a.`Current Agency Assignment Date Αναγκαστικής Προσημείωσης`   AS CurrentAgencyAssignmentDateForcedPrenotation
          ,a.`Current Stage Αναγκαστικής Προσημείωσης`                    AS CurrentStageForcedPrenotation
          ,a.`Current Stage Date Αναγκαστικής Προσημείωσης`               AS CurrentStageDateForcedPrenotation
          ,a.`Current Agency Επιβολής κατάσχεσης`                        AS CurrentAgencyConfiscation
          ,a.`Current Agency Assignment Date Επιβολής κατάσχεσης`         AS CurrentAgencyAssignmentDateconfiscation
          ,a.`Current Stage Επιβολής κατάσχεσης`                          AS CurrentStageconfiscation
          ,a.`Current Stage Date Επιβολής κατάσχεσης`                     AS CurrentStageDateconfiscation
          ,a.ON_BALANCE
          ,a.OFF_BALANCE
          ,a.TOTAL_BALANCE
          ,a.TOTAL_BALANCE_EMLT
          ,a.`Current Agency Τακτικής Αγωγής`                 AS CurrentAgencyRegularSuit
          ,a.`Current Agency Assignment Date Τακτικής Αγωγής` AS CurrentAgencyAssignmentDateRegularSuit
          ,a.`Current Stage Τακτικής Αγωγής`                  AS CurrentStageRegularSuit
          ,a.`Current Stage Date Τακτικής Αγωγής`             AS CurrentStageDateRegularSuit

     FROM `{env_prefix}-cdh`.landed.repointqcab_legalprocessassignment a
     LEFT JOIN NonCorePerimeter P ON p.ExposureKeyQCAlpha = a.PRIMARY_OWNER_ACCOUNT_NUMBER
     WHERE 1=1
          and  a.BusinessDate = {run_date!r}
       AND p.ExposureKeyQCAlpha IS NULL
'''
)

LegalProcessAssignment.createOrReplaceTempView("LegalProcessAssignment")

##### final_df

In [0]:
final_df = spark.sql(
f'''
SELECT 
      CAST(BusinessDate AS date) AS BusinessDate
    , CAST(CIF AS string) AS CIF
    , CAST(AFM AS string) AS AFM
    , CAST(CUST_LAST_NAME AS string) AS CUST_LAST_NAME
    , CAST(CUST_FIRST_NAME AS string) AS CUST_FIRST_NAME
    , CAST(CUST_FATH_NAME AS string) AS CUST_FATH_NAME
    , CAST(ACCOUNT AS string) AS ACCOUNT
    , CAST(PRIMARY_OWNER_ACCOUNT_NUMBER AS string) AS PRIMARY_OWNER_ACCOUNT_NUMBER
    , CAST(QC_RELATION_TYPE AS string) AS QC_RELATION_TYPE
    , CAST(ACCOUNT_STATUS AS string) AS ACCOUNT_STATUS
    , CAST(CURRENT_BUCKET AS string) AS CURRENT_BUCKET
    , CAST(CASE_OUTST_BALANCE AS string) AS CASE_OUTST_BALANCE
    , CAST(CASE_DEBT_AMOUNT AS string) AS CASE_DEBT_AMOUNT
    , CAST(DEBT_LPS AS string) AS DEBT_LPS
    , CAST(DEBT_LPS_DATE AS string) AS DEBT_LPS_DATE
    , CAST(CUS_LPS AS string) AS CUS_LPS
    , CAST(CurrentAgencyPaymentOrder AS string) AS CurrentAgencyPaymentOrder
    , CAST(CurrentAgencyAssignmentDate AS string) AS CurrentAgencyAssignmentDate
    , CAST(CurrentStagePaymentOrder AS string) AS CurrentStagePaymentOrder
    , CAST(CurrentStageDatePaymentOrder AS string) AS CurrentStageDatePaymentOrder
    , CAST(CurrentAgencyForcedPrenotation AS string) AS CurrentAgencyForcedPrenotation
    , CAST(CurrentAgencyAssignmentDateForcedPrenotation AS string) AS CurrentAgencyAssignmentDateForcedPrenotation
    , CAST(CurrentStageForcedPrenotation AS string) AS CurrentStageForcedPrenotation
    , CAST(CurrentStageDateForcedPrenotation AS string) AS CurrentStageDateForcedPrenotation
    , CAST(CurrentAgencyConfiscation AS string) AS CurrentAgencyConfiscation
    , CAST(CurrentAgencyAssignmentDateconfiscation AS string) AS CurrentAgencyAssignmentDateconfiscation
    , CAST(CurrentStageconfiscation AS string) AS CurrentStageconfiscation
    , CAST(CurrentStageDateconfiscation AS string) AS CurrentStageDateconfiscation
    , CAST(ON_BALANCE AS string) AS ON_BALANCE
    , CAST(OFF_BALANCE AS string) AS OFF_BALANCE
    , CAST(TOTAL_BALANCE AS string) AS TOTAL_BALANCE
    , CAST(TOTAL_BALANCE_EMLT AS string) AS TOTAL_BALANCE_EMLT
    , CAST(CurrentAgencyRegularSuit AS string) AS CurrentAgencyRegularSuit
    , CAST(CurrentAgencyAssignmentDateRegularSuit AS string) AS CurrentAgencyAssignmentDateRegularSuit
    , CAST(CurrentStageRegularSuit AS string) AS CurrentStageRegularSuit
    , CAST(CurrentStageDateRegularSuit AS string) AS CurrentStageDateRegularSuit
    , CAST(getdate() AS TIMESTAMP) AS InsertTimeStamp
FROM LegalProcessAssignment
'''
)
final_df.createOrReplaceTempView("final_df")

In [0]:
if final_df.rdd.isEmpty():
    raise ValueError(f"Core_LegalProcessAssignment: final DataFrame is empty - aborting write to {delivery_table_path} to avoid overwriting with no data")
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = ["ACCOUNT", "CIF"]
check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="warning")

In [0]:
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
LegalProcessAssignment = spark.sql(
f'''
SELECT * 
FROM {delivery_table_path}
WHERE BusinessDate = {run_date!r}
'''
)
LegalProcessAssignment.createOrReplaceTempView("LegalProcessAssignment")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/Core/LegalProcessAssignment/"
file_name = "LegalProcessAssignment_" +run_date+ ".csv"
LegalProcessAssignment.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}")
rename_and_cleanup_files(output_path, file_name)